# Qwen2.5-VL QLoRA — Multiple-Choice VQA on AI2D

Дообучение Qwen2.5-VL-3B на задаче выбора ответа из 4 вариантов (A/B/C/D).  
Формат: `[image + вопрос + 4 варианта]` → одна буква `A`/`B`/`C`/`D`.

**Ячейки:**
1. Bootstrap — найти корень проекта  
2. Конфиг — все параметры в одном месте  
3. Данные — загрузить манифест  
4. Dataset + Collator  
5. Модель + QLoRA  
6. Обучение  
7. Оценка VQA accuracy  
8. Сохранение метрик  

In [1]:
from pathlib import Path
import os, sys

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src' / 'vqa_retrieval').exists() and (p / 'experiments').exists():
            return p
    raise RuntimeError('Cannot find project root — открой ноутбук изнутри ai2d_vqa_clean')

ROOT     = find_project_root()
EXTERNAL = ROOT.parent
os.chdir(ROOT)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('ROOT:    ', ROOT)
print('EXTERNAL:', EXTERNAL)

ROOT:     C:\Users\Jet\Desktop\data\diagram_vqa
EXTERNAL: C:\Users\Jet\Desktop\data


In [2]:
# Все параметры здесь. Менять только эту ячейку.

MODEL_PATH = EXTERNAL / 'models' / 'Qwen2.5-VL-3B-Instruct'
MANIFEST   = EXTERNAL / 'ai2d' / 'prepared_v2' / 'manifest_hybrid.jsonl'
OUTPUT_DIR = ROOT / 'runs' / 'vlm_qlora_mcq'

# Обучение
EPOCHS     = 3        # кол-во эпох
LR         = 2e-4
BATCH_SIZE = 2        # батч на GPU (probe: RTX 4060 8GB тянет 2 при 512×512, пик 4469 MB)
GRAD_ACCUM = 4        # эффективный батч = 2 * 4 = 8 (то же что было, но ~2x быстрее)
MAX_LENGTH = 2048     # макс. токенов на пример

# Размер картинки: 262144 = 512×512 — максимум что влезает при BS=2 с запасом ~1.9 GB
IMAGE_MAX_PIXELS = 262144

LORA_R     = 16
LORA_ALPHA = 32

# None = полный датасет; ставь число для быстрой проверки (smoke run)
MAX_TRAIN  = None    # напр. 200 для smoke
MAX_VAL    = 200     # для eval

print('Model exists :', MODEL_PATH.exists())
print('Manifest exists:', MANIFEST.exists())
print('Output dir   :', OUTPUT_DIR)
print(f'Effective batch: {BATCH_SIZE} * {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')

Model exists : True
Manifest exists: True
Output dir   : C:\Users\Jet\Desktop\data\diagram_vqa\runs\vlm_qlora_mcq
Effective batch: 2 * 4 = 8


In [3]:
import json

LETTERS = ['A', 'B', 'C', 'D']

def build_prompt(question: str, options: list) -> str:
    """Строит MCQ-промпт: вопрос + 4 варианта."""
    opts = '\n'.join(f'{LETTERS[i]}) {opt}' for i, opt in enumerate(options))
    return (
        f"Look at the diagram and answer the question.\n\n"
        f"Question: {question}\n\n"
        f"{opts}\n\n"
        f"Answer with just the letter (A, B, C, or D)."
    )

def load_samples(manifest_path: Path, base_path: Path) -> dict:
    """Загружает манифест AI2D → {split: [samples]}.
    base_path — корень для резолва относительных путей картинок."""
    by_split = {'train': [], 'val': [], 'test': []}
    skipped = 0
    for line in manifest_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        r    = json.loads(line)
        sp   = r.get('split', 'train').lower()
        opts = r.get('options', [])
        idx  = r.get('correct_option_idx')
        # Путь может быть относительным → резолвим от base_path
        raw  = r['image_path']
        img  = Path(raw) if Path(raw).is_absolute() else base_path / raw
        if sp not in by_split or not opts or idx is None or idx >= len(opts) or not img.exists():
            skipped += 1
            continue
        by_split[sp].append({
            'image_path':   str(img),
            'question':     r['question'],
            'options':      opts,
            'label_idx':    int(idx),
            'label_letter': LETTERS[int(idx)],
        })
    for k, v in by_split.items():
        print(f'  {k}: {len(v)}')
    print(f'  skipped: {skipped}')
    return by_split

splits = load_samples(MANIFEST, base_path=EXTERNAL)

  train: 11145
  val: 1268
  test: 3088
  skipped: 0


In [4]:
import torch
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info


class MCQDataset(Dataset):
    """Простой Dataset — список сэмплов из манифеста."""
    def __init__(self, samples: list, max_samples=None):
        self.samples = samples[:max_samples] if max_samples else samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


class MCQCollator:
    """
    Собирает батч для обучения:
    - user:      картинка + MCQ-промпт
    - assistant: одна буква (A/B/C/D)
    - labels:    -100 на промпте, буква — цель обучения
    """
    def __init__(self, processor, max_length: int = 2048, image_max_pixels: int = 262144):
        self.processor       = processor
        self.max_length      = max_length
        self.image_max_pixels = image_max_pixels

    def __call__(self, batch: list) -> dict:
        user_msgs, full_msgs = [], []

        for item in batch:
            prompt   = build_prompt(item['question'], item['options'])
            # max_pixels ограничивает размер картинки → меньше visual tokens
            user_msg = {'role': 'user', 'content': [
                {'type': 'image', 'image': item['image_path'],
                 'max_pixels': self.image_max_pixels},
                {'type': 'text',  'text':  prompt},
            ]}
            asst_msg = {'role': 'assistant', 'content': [
                {'type': 'text', 'text': item['label_letter']},
            ]}
            user_msgs.append([user_msg])
            full_msgs.append([user_msg, asst_msg])

        prompt_texts = [
            self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in user_msgs
        ]
        full_texts = [
            self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in full_msgs
        ]

        all_images = []
        for m in user_msgs:
            imgs, _ = process_vision_info(m)
            all_images.extend(imgs or [])

        kw = dict(padding=True, truncation=True, max_length=self.max_length, return_tensors='pt')
        full_enc   = self.processor(text=full_texts,   images=all_images, **kw)
        prompt_enc = self.processor(text=prompt_texts, images=all_images, **kw)

        # Считаем loss только на ответе, промпт маскируем
        labels = full_enc['input_ids'].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        for i, plen in enumerate(prompt_enc['attention_mask'].sum(dim=1).tolist()):
            labels[i, :int(plen)] = -100
        full_enc['labels'] = labels
        return full_enc


print('MCQDataset и MCQCollator готовы.')

MCQDataset и MCQCollator готовы.


In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit NF4 квантизация — модель занимает ~2 GB вместо ~8 GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(str(MODEL_PATH), trust_remote_code=True, use_fast=False)

model = AutoModelForImageTextToText.from_pretrained(
    str(MODEL_PATH),
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache = False          # нужно для gradient checkpointing
model = prepare_model_for_kbit_training(model)

# LoRA — дообучаем только attention + FFN, всё остальное заморожено
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()      # должно быть ~1-2% от полного размера

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 37,152,768 || all params: 3,791,775,744 || trainable%: 0.9798


In [ ]:
import inspect
from transformers import Trainer, TrainingArguments, TrainerCallback
from IPython.display import display, HTML
import time

class LiveMetricsCallback(TrainerCallback):
    """Print live metrics every logging step."""
    def __init__(self):
        self._t0 = time.time()
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        loss = logs.get('loss')
        val_loss = logs.get('eval_loss')
        lr = logs.get('learning_rate')
        epoch = logs.get('epoch', state.epoch)
        step = state.global_step
        elapsed = (time.time() - self._t0) / 60
        parts = [f'step={step}', f'epoch={epoch:.2f}']
        if loss     is not None: parts.append(f'loss={loss:.4f}')
        if val_loss is not None: parts.append(f'val_loss={val_loss:.4f}')
        if lr       is not None: parts.append(f'lr={lr:.2e}')
        parts.append(f't={elapsed:.1f}m')
        print('  '.join(parts), flush=True)


train_ds = MCQDataset(splits['train'], max_samples=MAX_TRAIN)
val_ds   = MCQDataset(splits['val'],   max_samples=MAX_VAL)
collator = MCQCollator(processor, max_length=MAX_LENGTH, image_max_pixels=IMAGE_MAX_PIXELS)
print(f'Train: {len(train_ds)},  Val: {len(val_ds)}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Совместимость с разными версиями transformers
ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_key  = 'evaluation_strategy' if 'evaluation_strategy' in ta_params else 'eval_strategy'

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = 0.03,
    weight_decay                = 0.01,
    bf16                        = torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16                        = torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    logging_steps               = 1,
    logging_first_step          = True,
    save_strategy               = 'epoch',
    save_total_limit            = 1,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    remove_unused_columns       = False,
    report_to                   = [],
    dataloader_num_workers      = 0,    # 0 = main process; иначе segfault на Windows/Linux с CUDA
    **{eval_key: 'epoch'},
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    data_collator = collator,
    callbacks     = [LiveMetricsCallback()],
)

train_result = trainer.train()
print('\nTrain loss:', round(train_result.training_loss, 4))

Train: 11145,  Val: 200


C:\Users\Jet\Desktop\data\.venv\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
C:\Users\Jet\Desktop\data\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\Jet\Desktop\d

Epoch,Training Loss,Validation Loss


step=1  epoch=0.00  loss=0.1712  lr=1.59e-06  t=16.2m
step=2  epoch=0.00  loss=0.1267  lr=3.17e-06  t=30.8m
step=3  epoch=0.00  loss=0.1292  lr=4.76e-06  t=48.6m
step=4  epoch=0.00  loss=0.2025  lr=6.35e-06  t=63.3m
step=5  epoch=0.00  loss=0.1984  lr=7.94e-06  t=79.0m
step=6  epoch=0.00  loss=0.1907  lr=9.52e-06  t=99.0m
step=7  epoch=0.01  loss=0.3269  lr=1.11e-05  t=123.1m
step=8  epoch=0.01  loss=0.1970  lr=1.27e-05  t=142.6m
step=9  epoch=0.01  loss=0.2264  lr=1.43e-05  t=163.6m
step=10  epoch=0.01  loss=0.4281  lr=1.59e-05  t=182.8m
step=11  epoch=0.01  loss=0.2511  lr=1.75e-05  t=203.9m
step=12  epoch=0.01  loss=0.1476  lr=1.90e-05  t=221.7m
step=13  epoch=0.01  loss=0.0800  lr=2.06e-05  t=240.3m
step=14  epoch=0.01  loss=0.2455  lr=2.22e-05  t=257.3m
step=15  epoch=0.01  loss=0.4126  lr=2.38e-05  t=277.6m
step=16  epoch=0.01  loss=0.1430  lr=2.54e-05  t=300.2m
step=17  epoch=0.01  loss=0.2484  lr=2.70e-05  t=318.8m
step=18  epoch=0.01  loss=0.1871  lr=2.86e-05  t=340.5m
step=19

In [ ]:
adapter_path = OUTPUT_DIR / 'adapter'
trainer.save_model(str(adapter_path))
processor.save_pretrained(str(adapter_path))
print('Адаптер сохранён:', adapter_path)

In [ ]:
# Прогоняем модель на val/test — смотрим сколько букв совпало.

def evaluate_vqa(model, processor, samples: list, max_samples: int = None, label: str = '') -> dict:
    model.eval()
    device  = next(model.parameters()).device
    correct = 0
    total   = len(samples) if max_samples is None else min(len(samples), max_samples)

    for item in samples[:total]:
        msgs = [{'role': 'user', 'content': [
            {'type': 'image', 'image': item['image_path']},
            {'type': 'text',  'text':  build_prompt(item['question'], item['options'])},
        ]}]
        text    = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, _ = process_vision_info(msgs)
        inputs  = processor(text=[text], images=imgs, return_tensors='pt')
        inputs  = {k: v.to(device) for k, v in inputs.items() if hasattr(v, 'to')}

        with torch.inference_mode():
            out_ids = model.generate(**inputs, max_new_tokens=5)
        output = processor.decode(
            out_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()

        pred = output[0].upper() if output else ''
        if pred == item['label_letter']:
            correct += 1

    acc    = correct / max(1, total)
    result = {'vqa_accuracy': round(acc, 4), 'correct': correct, 'total': total}
    tag    = f'[{label}] ' if label else ''
    print(f'{tag}VQA accuracy: {acc:.4f}  ({correct}/{total})')
    return result


val_metrics  = evaluate_vqa(model, processor, splits['val'],  label='val')
test_metrics = evaluate_vqa(model, processor, splits['test'], label='test')